# **Preprocessing T1 and DTI images for MCI classification**


---

For the T1 data (tabular), we select the features we want to use, correct covariates and do a Z-score normalization.

For the DTI data (graphs), we select which edge features to keep and do a Z-score normalization.

The T1 data are eventually fused into the graphs as node features.

This is done across 5 folds, where each fold consists of a train/val set (80%) and a test set (20%). Each train/val set is then split into separate train and val sets (85/15 split on the initial 80%).

All compuations for covariate corrections and normalizations are based on the CN subjects of the train set within each fold. This is done to ensure unbiased cross validation during model training. Furthermore, the train set is used to perform an inner (nested) 3-fold validation, in order to perform hyperparameter tuning during training.

---

After all cells are ran in order, the notebook produces 5 train, 5 val and 5 test sets, one per fold, as well as a list that determines the inner fold splits by subject ID's.

In [ ]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pandas as pd
import os
import re
import copy
import warnings
import pickle
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.linear_model import LinearRegression
import networkx as nx
import numpy as np
from tqdm import tqdm
import torch
from torch_geometric.data import Data

warnings.filterwarnings('ignore')

## Load and prepare data for preprocessing

In [ ]:
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/team3_xai_gnn'
smri_path = os.path.join(base_path, 'oasis3_baseline_freesurfer_dataset.csv')
dmri_path = os.path.join(base_path, 'oasis3_graphmls_scale1')
output_dir = os.path.join(base_path, 'preprocessed')
os.makedirs(output_dir, exist_ok=True)

df_raw = pd.read_csv(smri_path)
graphml_files = [f for f in os.listdir(dmri_path) if f.endswith('.graphml')]

Mounted at /content/drive


Filter out subjects that don't have both T1 and DTI data

In [ ]:
def parse_filename(fname):
    match = re.search(r'sub-(OAS3\d+)_ses-(d\d+)', fname)
    if match:
        return match.group(1), match.group(2)
    return None, None

csv_pairs = set(zip(df_raw['subject_id'], df_raw['session_id']))

accepted = []
for f in graphml_files:
    subj, ses = parse_filename(f)
    if (subj, ses) in csv_pairs:
        accepted.append(f)

Drop columns from T1 data and make a list of DTI edges

In [ ]:
# drop these columns from T1 data
cols_to_drop = [
    '5th-Ventricle_volume', 'CSF_volume', 'CortexVol', 'CorticalWhiteMatterVol',
    'IntraCranialVol', 'Optic-Chiasm_volume', 'SubCortGrayVol', 'SupraTentorialVol',
    'TOTAL_HIPPOCAMPUS_VOLUME', 'TotalGrayVol', 'WM-hypointensities_volume',
    'lhCortexVol', 'lhCorticalWhiteMatterVol', 'non-WM-hypointensities_volume',
    'rhCortexVol', 'rhCorticalWhiteMatterVol', 'Right-non-WM-hypointensities_volume',
    'Left-non-WM-hypointensities_volume', 'Right-WM-hypointensities_volume',
    'Left-WM-hypointensities_volume', 'L.Numvert', 'R.NumVert', 'L.SurfArea', 'R.SurfArea'
]

# keep a copy since we actually want to drop icv eventually
icv = df_raw['IntraCranialVol'].copy()
# remove cols and drop dementia class
df = df_raw.drop(columns=cols_to_drop)
df = df[df['label'] != 2].reset_index(drop=True)
icv = icv.iloc[df.index].reset_index(drop=True)
df['icv'] = icv.values

# identify volume and thickness columns
meta_cols = ['subject_id', 'session_id', 'label', 'age', 'gender', 'icv']
thickness_cols = [c for c in df.columns if 'thickness' in c.lower()]
volume_cols = [c for c in df.columns if c not in meta_cols and 'thickness' not in c.lower()]

# edges to keep from DTI data (fiber_density dropped)
edge_feature_names = [
    'number_of_fibers', 'fiber_length_mean', 'fiber_length_median', 'fiber_length_std',
    'fiber_proportion', 'normalized_fiber_density',
    'shore_gfa_mean', 'shore_gfa_std', 'shore_gfa_median',
    'shore_msd_mean', 'shore_msd_std', 'shore_msd_median',
    'shore_rtop_signal_mean', 'shore_rtop_signal_std', 'shore_rtop_signal_median'
]

Load graphs

In [ ]:
def load_graph(fname):
    '''
    loads graphs and removes self edges
    '''
    path = os.path.join(dmri_path, fname)
    G = nx.read_graphml(path)
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return fname, G

graphs = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    for fname, G in tqdm(executor.map(load_graph, accepted), total=len(accepted), desc='Loading graphs'):
        graphs[fname] = G

Loading graphs: 100%|██████████| 676/676 [05:40<00:00,  1.99it/s]


Create outer 5-fold sets

In [ ]:
# 5-fold outer split
X_dummy = np.arange(len(df))
y       = df['label'].values
groups  = df['subject_id'].values

skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_ids = []
for train_val_idx, test_idx in skf.split(X_dummy, y, groups):
    # train/val vs test split
    train_val_df = df.iloc[train_val_idx]
    test_df      = df.iloc[test_idx]

    # train vs val split
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=0.15,
        stratify=train_val_df['label'],
        random_state=42
    )

    # create a list of the subject IDs in each set, for each fold
    fold_ids.append({
        'train': set(train_df['subject_id']),
        'val':   set(val_df['subject_id']),
        'test':  set(test_df['subject_id']),
    })

## Preprocess data

Start with preprocessing T1 data

In [ ]:
def preprocess_fold(fold_idx):
    '''
    performs preprocessing for T1 data on a given fold
    stores mean/std values for edge preprocessing

    uses the subject IDs appointed to each set for this fold
    '''

    # get subject IDs for this fold
    ids = fold_ids[fold_idx]
    # create the train/val/test sets
    train_df = df[df['subject_id'].isin(ids['train'])].copy()
    val_df   = df[df['subject_id'].isin(ids['val'])].copy()
    test_df  = df[df['subject_id'].isin(ids['test'])].copy()

    # use only CN subjects from the train set
    cn_train_df = train_df[train_df['label'] == 0].copy()

    # --- covariate correction
    age_bins = [(42, 69), (70, 96)]

    for feature_cols, predictors in [
        (volume_cols,    ['age', 'gender', 'icv']),
        (thickness_cols, ['age', 'gender'])
    ]:
        for low, high in age_bins:
            cn_mask = (cn_train_df['age'] >= low) & (cn_train_df['age'] <= high)
            cn_bin  = cn_train_df[cn_mask]
            if len(cn_bin) == 0:
                continue
            X_fit = cn_bin[predictors].values
            for col in feature_cols:
                model = LinearRegression()
                model.fit(X_fit, cn_bin[col].values)
                for split_df in [train_df, val_df, test_df]:
                    mask  = (split_df['age'] >= low) & (split_df['age'] <= high)
                    X_pred = split_df.loc[mask, predictors].values
                    split_df.loc[mask, col] = split_df.loc[mask, col].values - model.predict(X_pred)

    # --- z-score normalization of node features
    cn_corrected = train_df[train_df['label'] == 0]

    vol_mean = cn_corrected[volume_cols].mean().values
    vol_std  = cn_corrected[volume_cols].std().values
    vol_std[vol_std == 0] = 1e-8

    tck_mean = cn_corrected[thickness_cols].mean().values
    tck_std  = cn_corrected[thickness_cols].std().values
    tck_std[tck_std == 0] = 1e-8

    for split_df in [train_df, val_df, test_df]:
        split_df.loc[:, volume_cols] = (split_df[volume_cols].values - vol_mean) / vol_std
        split_df.loc[:, thickness_cols] = (split_df[thickness_cols].values - tck_mean) / tck_std

    # --- z-score normalization of edge features
    # edge normalization is not performed here
    # just compute the mean/std and save it
    cn_train_subjects = set(cn_train_df['subject_id'])
    cn_train_fnames = [f for f in accepted if parse_filename(f)[0] in cn_train_subjects]

    cn_edge_attrs = []
    for fname in cn_train_fnames:
        G = graphs[fname]
        for _, _, data in G.edges(data=True):
            cn_edge_attrs.append([float(data.get(feat, 0.0)) for feat in edge_feature_names])

    cn_edge_attrs = np.array(cn_edge_attrs)
    edge_mean = cn_edge_attrs.mean(axis=0)
    edge_std  = cn_edge_attrs.std(axis=0)
    edge_std[edge_std == 0] = 1e-8

    # store results
    return {
        'train':     train_df,
        'val':       val_df,
        'test':      test_df,
        'edge_mean': edge_mean,
        'edge_std':  edge_std,
    }

# run preprocessing function for every outer fold
fold_data = [preprocess_fold(i) for i in range(5)]

Define functions for preprocessing DTI graphs

In [ ]:
def get_node_features(node_attrs, row, volume_cols, thickness_cols):
    '''
    returns [volume, thickness, has_volume, has_thickness] for a node.
    - volume and thickness are the z-score corrected values from the sMRI row.
    - has_volume / has_thickness are 1.0 if the column exists for this node, 0.0 otherwise.
    '''

    raw_fsname   = node_attrs.get('dn_fsname',    '').strip()
    raw_hemi     = node_attrs.get('dn_hemisphere', '').strip()
    raw_region   = node_attrs.get('dn_region',     '').strip()
    raw_name     = node_attrs.get('dn_name',       '').strip()

    region = ''
    if raw_region in ['cortical', 'subcortical']:
        region = raw_region
    elif 'ctx-' in raw_name or 'ctx-' in raw_fsname:
        region = 'cortical'
    elif raw_fsname.startswith(('Left-', 'Right-')) or raw_name.startswith(('Left-', 'Right-')) or raw_fsname == 'Brain-Stem':
        region = 'subcortical'

    vol_val, tck_val, has_vol, has_tck = 0.0, 0.0, 0.0, 0.0

    if region == 'cortical':
        anat_name = ''
        if raw_hemi and 'ctx-' in raw_fsname:
            parts = raw_fsname.split('-', 2)
            if len(parts) == 3:
                anat_name = f'{parts[1]}_{parts[2]}'
        elif 'ctx-' in raw_name:
            parts = raw_name.split('-', 2)
            if len(parts) == 3:
                anat_name = f'{parts[1]}_{parts[2]}'
        elif raw_hemi and raw_fsname:
            anat_name = f'{raw_hemi}_{raw_fsname}'

        vol_col = f'{anat_name}_volume'
        tck_col = f'{anat_name}_thickness'
        has_vol = 1.0 if vol_col in row.index else 0.0
        has_tck = 1.0 if tck_col in row.index else 0.0
        vol_val = float(row[vol_col]) if has_vol else 0.0
        tck_val = float(row[tck_col]) if has_tck else 0.0

    elif region == 'subcortical':
        anat_name = ''
        if raw_fsname and raw_fsname not in ('subcortical', ''):
            anat_name = raw_fsname.replace('_area', '-area')
        elif raw_name and raw_name not in ('subcortical', ''):
            anat_name = raw_name.replace('_area', '-area')

        vol_col = f'{anat_name}_volume'
        has_vol = 1.0 if vol_col in row.index else 0.0
        vol_val = float(row[vol_col]) if has_vol else 0.0

    return [vol_val, tck_val, has_vol, has_tck]


def build_pyg_graph(fname, meta_row, edge_mean, edge_std):
    '''
    converts a NetworkX graph + one sMRI metadata row into a PyG Data object
    node features: [volume, thickness, has_volume, has_thickness]
    edge features: z-score normalised edge attributes
    '''

    G = graphs[fname]
    nodes   = list(G.nodes(data=True))
    node_map = {nid: i for i, (nid, _) in enumerate(nodes)}

    volume_cols_set    = set(volume_cols)
    thickness_cols_set = set(thickness_cols)

    node_features = []
    for _, attrs in nodes:
        feats = get_node_features(attrs, meta_row, volume_cols_set, thickness_cols_set)
        node_features.append(feats)

    edge_index = []
    edge_attr  = []
    for u, v, edata in G.edges(data=True):
        ui, vi = node_map[u], node_map[v]
        feats  = np.array([float(edata.get(feat, 0.0)) for feat in edge_feature_names])
        normed = (feats - edge_mean) / edge_std
        # bidirectional
        edge_index += [[ui, vi], [vi, ui]]
        edge_attr  += [normed, normed]

    x           = torch.tensor(node_features, dtype=torch.float)
    edge_index  = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr   = torch.tensor(np.array(edge_attr), dtype=torch.float)
    y           = torch.tensor([int(meta_row['label'])], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)


def build_graph_list(split_df, edge_mean, edge_std):
    '''
    builds a list of PyG Data objects for all subjects in a split DataFrame
    '''

    subj_to_fname = {parse_filename(f)[0]: f for f in accepted}
    data_list = []
    for _, row in split_df.iterrows():
        subj = row['subject_id']
        if subj in subj_to_fname:
            g = build_pyg_graph(subj_to_fname[subj], row, edge_mean, edge_std)
            data_list.append(g)
    return data_list

Create final graphs and save them

In [ ]:
# --- build and save outer fold graphs
for fold_idx in range(5):
    fd         = fold_data[fold_idx]
    edge_mean  = fd['edge_mean']
    edge_std   = fd['edge_std']

    train_graphs_list = build_graph_list(fd['train'], edge_mean, edge_std)
    val_graphs_list   = build_graph_list(fd['val'],   edge_mean, edge_std)
    test_graphs_list  = build_graph_list(fd['test'],  edge_mean, edge_std)

    torch.save(train_graphs_list, os.path.join(output_dir, f'fold_{fold_idx+1}_train.pt'))
    torch.save(val_graphs_list,   os.path.join(output_dir, f'fold_{fold_idx+1}_val.pt'))
    torch.save(test_graphs_list,  os.path.join(output_dir, f'fold_{fold_idx+1}_test.pt'))

print('Outer folds saved.')

Outer folds saved.


Make a list with the subject ID's for the inner 3-fold loop

In [ ]:
# --- build inner 3-fold splits on each outer train set
# inner splits are saved as subject ID lists only (not preprocessed graphs),
# since hyperparameter tuning models will load the outer train graphs and index into them.

inner_fold_ids = []  # list of 5 (one per outer fold), each is a list of 3 dicts with 'inner_train', 'inner_val'

inner_skf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

for fold_idx in range(5):
    train_df   = fold_data[fold_idx]['train']
    X_inner    = np.arange(len(train_df))
    y_inner    = train_df['label'].values
    grp_inner  = train_df['subject_id'].values

    inner_folds = []
    for inner_train_idx, inner_val_idx in inner_skf.split(X_inner, y_inner, grp_inner):
        inner_train_subs = set(train_df.iloc[inner_train_idx]['subject_id'])
        inner_val_subs   = set(train_df.iloc[inner_val_idx]['subject_id'])
        inner_folds.append({
            'inner_train': inner_train_subs,
            'inner_val':   inner_val_subs,
        })
    inner_fold_ids.append(inner_folds)

# save inner fold subject ID lists
with open(os.path.join(output_dir, 'inner_fold_ids.pkl'), 'wb') as f:
    pickle.dump(inner_fold_ids, f)

print('Inner folds saved.')

Inner folds saved.
